In [ ]:
import os
import joblib
import pandas as pd
import tkinter as tk
from tkinter import ttk, scrolledtext, messagebox

MODEL_FILE = "best_lr_grid_bow.pkl"
VECTORIZER_FILE = "bow.pkl"

try:
    model = joblib.load(MODEL_FILE)
    vectorizer = joblib.load(VECTORIZER_FILE)
except Exception as e:
    raise SystemExit(f"Error loading model/vectorizer:\n{e}")

root = tk.Tk()
root.title("Fake News Detection Prototype")
root.state("zoomed")
root.minsize(1200, 800)
root.configure(bg="#EEF4FB")

style = ttk.Style(root)
style.theme_use("clam")
style.configure("TProgressbar", thickness=18)

feedback = tk.StringVar()

def predict():
    article = txt.get("1.0", "end").strip()

    if not article:
        messagebox.showwarning("Warning", "Please enter a news article.")
        return

    try:
        X = vectorizer.transform([article])
        pred = model.predict(X)[0]
        prob = model.predict_proba(X)[0].max()*100

        if pred == 0:
            badge.config(text="FAKE NEWS", bg="#D32F2F")
        else:
            badge.config(text="REAL NEWS", bg="#2E7D32")

        confidence.config(text=f"{prob:.2f}%")
        bar["value"] = prob
        status.config(text="Prediction completed successfully.")

    except Exception as e:
        messagebox.showerror("Prediction Error", str(e))

def reset():
    txt.delete("1.0","end")
    badge.config(text="WAITING", bg="#607D8B")
    confidence.config(text="0%")
    bar["value"]=0
    feedback.set("")
    status.config(text="Ready")

def save_feedback():
    article=txt.get("1.0","end").strip()
    if not article:
        messagebox.showwarning("Warning","Enter a news article.")
        return

    df=pd.DataFrame({
        "Article":[article],
        "Prediction":[badge["text"]],
        "Confidence":[confidence["text"]],
        "Feedback":[feedback.get()]
    })

    file="Human_Evaluation_Feedback.csv"

    if os.path.exists(file):
        df.to_csv(file,mode="a",header=False,index=False)
    else:
        df.to_csv(file,index=False)

    messagebox.showinfo("Saved","Feedback saved successfully.")

header=tk.Frame(root,bg="#0D47A1",height=90)
header.pack(fill="x")

tk.Label(header,text="FAKE NEWS DETECTION PROTOTYPE",
font=("Segoe UI",24,"bold"),bg="#0D47A1",fg="white").pack(pady=(15,2))

tk.Label(header,text="Machine Learning Based News Classification",
font=("Segoe UI",11),bg="#0D47A1",fg="white").pack()

container=tk.Frame(root,bg="#EEF4FB")
container.pack(fill="both",expand=True,padx=20,pady=20)

container.grid_columnconfigure(0,weight=3)
container.grid_columnconfigure(1,weight=1)
container.grid_rowconfigure(0,weight=1)

left=tk.LabelFrame(container,text="News Article",
font=("Segoe UI",12,"bold"),bg="white")
left.grid(row=0,column=0,sticky="nsew",padx=(0,15))

txt=scrolledtext.ScrolledText(left,font=("Segoe UI",11),wrap="word")
txt.pack(fill="both",expand=True,padx=10,pady=10)

right=tk.LabelFrame(container,text="Prediction Dashboard",
font=("Segoe UI",12,"bold"),bg="white")
right.grid(row=0,column=1,sticky="nsew")

tk.Label(right,text="Prediction",font=("Segoe UI",13,"bold"),bg="white").pack(pady=(20,5))

badge=tk.Label(right,text="WAITING",font=("Segoe UI",18,"bold"),
bg="#607D8B",fg="white",width=18,pady=10)
badge.pack()

tk.Label(right,text="Confidence",font=("Segoe UI",12,"bold"),
bg="white").pack(pady=(30,5))

confidence=tk.Label(right,text="0%",font=("Segoe UI",24,"bold"),
fg="#1565C0",bg="white")
confidence.pack()

bar=ttk.Progressbar(right,maximum=100,length=250)
bar.pack(pady=10)

tk.Label(right,text="Human Evaluation",
font=("Segoe UI",12,"bold"),bg="white").pack(pady=(30,10))

ttk.Radiobutton(right,text="Prediction Correct",
variable=feedback,value="Correct").pack(anchor="w",padx=35)

ttk.Radiobutton(right,text="Prediction Incorrect",
variable=feedback,value="Incorrect").pack(anchor="w",padx=35)

buttons=tk.Frame(root,bg="#EEF4FB")
buttons.pack(fill="x",padx=20,pady=10)

for txt_btn,cmd,color in [
("Predict",predict,"#1565C0"),
("Submit Feedback",save_feedback,"#2E7D32"),
("Reset",reset,"#EF6C00"),
("Exit",root.destroy,"#C62828")]:
    tk.Button(buttons,text=txt_btn,command=cmd,bg=color,fg="white",
              font=("Segoe UI",11,"bold"),width=18,height=2).pack(side="left",padx=12)

status=tk.Label(root,text="Ready",anchor="w",bg="#CFD8DC",
font=("Segoe UI",10))
status.pack(fill="x",side="bottom")

root.mainloop()
